In [1]:
from pandas import read_excel
from statsmodels.formula.api import ols
import statsmodels.api as sm
import math
import numpy as np

In [ ]:
df =  read_excel("data-for-regression-excel.xlsx", sheet_name="data")
df.columns = [col.replace(' ', '') for col in df.columns]
df.columns = [col.replace(' ', '') for col in df.columns]
df.head()

In [3]:
# Hồi quy tuyến tính
model = ols("Price ~ Area + AccessRoad + Bedrooms", data = df).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  Price   R-squared:                       0.212
Model:                            OLS   Adj. R-squared:                  0.211
Method:                 Least Squares   F-statistic:                     237.2
Date:                Wed, 23 Apr 2025   Prob (F-statistic):          2.69e-136
Time:                        17:40:30   Log-Likelihood:                -5571.3
No. Observations:                2645   AIC:                         1.115e+04
Df Residuals:                    2641   BIC:                         1.117e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      2.2611      0.128     17.604      0.000       2.009       2.513
Area           0.0040      0.001      5.003      0.000       0.002       0.006
AccessRoad     0.0477      0.005      8.877      0.000       0.037       0.058
Bedrooms       0.7299      0.033     22.106      0.000       0.665       0.795
==============================================================================
Omnibus:                       49.993   Durbin-Watson:                   1.844
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               41.429
Skew:                           0.236   Prob(JB):                     1.01e-09
Kurtosis:                       2.608   Cond. No.                         296.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

$$
M(\mathbf{w}, \mathbf{x}) = w_1 \cdot \log(x_1 + 1) + w_2 x_2 + w_3 x_3 + w_4
$$

In [4]:
# Chuẩn bị dữ liệu X (list of lists) và y
X = df[["Area", "AccessRoad", "Bedrooms"]].values.tolist()  # x1, x2, x3
y = df["Price"].tolist()
m = len(X)

In [5]:
# Hàm mục tiêu
def compute_F(w):
    total = 0
    for i in range(m):
        x = X[i]
        prediction = w[0] * math.log(x[0] + 1) + w[1] * x[1] + w[2] * x[2] + w[3]
        residual = y[i] - prediction
        total += residual * residual
    return total / 2

# Gradient
def compute_gradient(w):
    grad = [0, 0, 0, 0]
    for i in range(m):
        x = X[i]
        log_area = math.log(x[0] + 1)
        prediction = w[0] * log_area + w[1] * x[1] + w[2] * x[2] + w[3]
        residual = y[i] - prediction
        grad[0] += -residual * log_area
        grad[1] += -residual * x[1]
        grad[2] += -residual * x[2]
        grad[3] += -residual
    return grad

# Line search
def line_search(w, grad):
    alpha_values = [0.000001, 0.000005, 0.00001, 0.00005, 0.0001]
    best_alpha = 0
    best_F = compute_F(w)
    for alpha in alpha_values:
        w_new = [w[j] - alpha * grad[j] for j in range(4)]
        F_new = compute_F(w_new)
        if F_new < best_F:
            best_F = F_new
            best_alpha = alpha
    return best_alpha

# Gradient descent
def gradient_descent(w0, k_max=10000, tolerance=0.01):
    w = w0[:]
    k = 0
    while k < k_max:
        grad = compute_gradient(w)
        grad_norm = sum([grad[j] * grad[j] for j in range(4)]) ** 0.5
        if grad_norm < tolerance:
            print(f"Đã hội tụ sau {k} bước. Gradient norm: {grad_norm}")
            break
        alpha = line_search(w, grad)
        if alpha == 0:
            print("Không tìm được alpha phù hợp. Dừng lại.")
            break
        for j in range(4):
            w[j] -= alpha * grad[j]
        if k % 1000 == 0 or k >= k_max - 5:
            print(f"Bước {k}: w = {w}, F(w) = {compute_F(w)}")
        k += 1
    return w

In [6]:
w0 = [0, 0, 0, 0]
w_opt = gradient_descent(w0)

print(f"w1 = {w_opt[0]}, w2 = {w_opt[1]}, w3 = {w_opt[2]}, w4 = {w_opt[3]}")
print(f"Giá trị F tại nghiệm tối ưu: {compute_F(w_opt)}")

Bước 0: w = [0.060098261051992535, 0.11992880549999996, 0.05291095000000004, 0.014401390000000005], F(w) = 28422.461675550523
Bước 1000: w = [0.6035929052919662, 0.04166050679085741, 0.7166839865529336, 0.15019263300893956], F(w) = 5167.773341868608
Bước 2000: w = [0.6041827900521453, 0.041649815561659434, 0.71674200712216, 0.1476050694709924], F(w) = 5167.772186466961
Bước 3000: w = [0.6046613487381067, 0.041642538938195556, 0.716789302055855, 0.14550827513715178], F(w) = 5167.771426343485
Bước 4000: w = [0.605049459155056, 0.04163555562959866, 0.7168274024814703, 0.14380620021132665], F(w) = 5167.770926365077
Bước 5000: w = [0.60536420196895, 0.041629873125983326, 0.7168582959178302, 0.14242585391384643], F(w) = 5167.770597546898
Bước 6000: w = [0.6056194446527471, 0.04162524353813287, 0.7168833441065006, 0.1413064222218026], F(w) = 5167.770381300419
Bước 7000: w = [0.6058264873180023, 0.041622117380972816, 0.7169038108871895, 0.14039930140777987], F(w) = 5167.770239023346
Bước 8000: